In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "BPQ_F.xpt"
csv_file = DATA_DIR / "BPQ_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched source values.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 BPQ_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BPQ020",
    "BPQ030",
    "BPD035",
    "BPQ040A",
    "BPQ050A",
    "BPQ057",
    "BPQ056",
    "BPD058",
    "BPQ059",
    "BPQ060",
    "BPQ070",
    "BPQ080",
    "BPQ090A",
    "BPQ090B",
    "BPQ090C",
    "BPQ090D",
    "BPQ100A",
    "BPQ100B",
    "BPQ100C",
    "BPQ100D"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "BPQ_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 6889:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 21:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,889 rows x 21 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# Never globally replace this number before
# validating its variable and frequency.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Attached BPQ_F contains:
#
#     BPD058 = 4 tiny values
#
# CDC defines BPD058 valid range:
#
#     0 to 1461
#
# so zero is legitimate.
#

EXPECTED_TINY_COUNTS = {
    "BPD058": 4
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated BPQ_F pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 4:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total}"
    )


print(
    "\nPASS: Tiny-value pattern matches "
    "validated BPQ_F pattern."
)


# ============================================================
# 6. RESTORE VALIDATED BPD058 ZEROS
# ============================================================

#
# Restore ONLY the four validated BPD058 values.
#
# Do not globally replace tiny values.
#

tiny_mask = (
    df["BPD058"]
    .eq(TINY_VALUE)
)


correction_count = int(
    tiny_mask.sum()
)


if correction_count != 4:

    raise ValueError(
        "Unexpected BPD058 tiny-value count."
    )


df.loc[
    tiny_mask,
    "BPD058"
] = 0


print("\n--- ZERO RESTORATION ---")


print(
    "BPD058 values restored to zero:",
    correction_count
)


if int(
    df["BPD058"]
    .eq(0)
    .sum()
) != 4:

    raise ValueError(
        "BPD058 zero restoration failed."
    )


print(
    "PASS: 4 validated BPD058 zeros restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# file should contain.
#
# The ONLY expected change relative to raw pandas XPT:
#
#     BPD058 tiny value -> 0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["BPD058"].eq(TINY_VALUE),
    "BPD058"
] = 0


# ============================================================
# 8. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# Never round genuine decimal values.
#
# This check occurs before any Int64 conversion.
#
# BPQ_F contains no genuine fractional values.
#
# If a future file does, STOP rather than rounding.
#
# This same safeguard protects Dietary decimal
# variables and continuous Lab / Examination values.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"Genuine fractional values found in {col}:",
            int(
                fractional_mask.sum()
            )
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimals detected. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in BPQ_F."
)


# ============================================================
# 9. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 10. FORMAT QUESTIONNAIRE VALUES WITHOUT .0
# ============================================================

#
# After:
#
#   - validated zero restoration
#   - genuine-decimal check
#
# every BPQ_F variable is proven whole-number.
#
# Therefore all can be Int64.
#
# IMPORTANT:
#
# NO .round() is used.
#

for col in df.columns:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All BPQ_F fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 11. CDC CATEGORICAL FREQUENCY CHECKS
# ============================================================

#
# Expected frequencies from CDC BPQ_F codebook.
#
# Codes with published frequency 0 are omitted from
# dictionaries because pandas value_counts() does not
# report absent values.
#

CDC_CATEGORICAL = {

    "BPQ020": {
        "values": {
            1: 2221,
            2: 4655,
            9: 13
        },
        "missing": 0
    },

    "BPQ030": {
        "values": {
            1: 1791,
            2: 418,
            9: 12
        },
        "missing": 4668
    },

    "BPQ040A": {
        "values": {
            1: 1937,
            2: 284
        },
        "missing": 4668
    },

    "BPQ050A": {
        "values": {
            1: 1742,
            2: 195
        },
        "missing": 4952
    },

    "BPQ057": {
        "values": {
            1: 963,
            2: 5878,
            9: 48
        },
        "missing": 0
    },

    "BPQ056": {
        "values": {
            1: 1491,
            2: 5397,
            9: 1
        },
        "missing": 0
    },

    "BPQ059": {
        "values": {
            1: 775,
            2: 6114
        },
        "missing": 0
    },

    "BPQ060": {
        "values": {
            1: 4358,
            2: 1705,
            7: 1,
            9: 154
        },
        "missing": 671
    },

    "BPQ070": {
        "values": {
            1: 2940,
            2: 734,
            3: 445,
            4: 221,
            9: 18
        },
        "missing": 2531
    },

    "BPQ080": {
        "values": {
            1: 1875,
            2: 2449,
            9: 34
        },
        "missing": 2531
    },

    "BPQ090A": {
        "values": {
            1: 1552,
            2: 320,
            9: 3
        },
        "missing": 5014
    },

    "BPQ090B": {
        "values": {
            1: 1189,
            2: 681,
            9: 5
        },
        "missing": 5014
    },

    "BPQ090C": {
        "values": {
            1: 1373,
            2: 497,
            9: 5
        },
        "missing": 5014
    },

    "BPQ090D": {
        "values": {
            1: 1335,
            2: 539,
            9: 1
        },
        "missing": 5014
    },

    "BPQ100A": {
        "values": {
            1: 1304,
            2: 248
        },
        "missing": 5337
    },

    "BPQ100B": {
        "values": {
            1: 1000,
            2: 188,
            9: 1
        },
        "missing": 5700
    },

    "BPQ100C": {
        "values": {
            1: 1092,
            2: 281
        },
        "missing": 5516
    },

    "BPQ100D": {
        "values": {
            1: 1136,
            2: 198,
            9: 1
        },
        "missing": 5554
    }
}


print("\n--- CDC CATEGORICAL FREQUENCY CHECKS ---")


for col, expected in CDC_CATEGORICAL.items():

    actual_values = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_values != expected["values"]:

        raise ValueError(
            f"{col} frequencies differ from CDC.\n"
            f"Expected: {expected['values']}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected["missing"]:

        raise ValueError(
            f"{col} missing count differs from CDC.\n"
            f"Expected: {expected['missing']}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 12. CDC CHECK - BPD035
# ============================================================

#
# Age first told participant had hypertension.
#
# CDC:
#
# 5-79 = 2,130
# 80    =    41
# 777   =     1
# 999   =    40
# Missing = 4,677
#

s = df["BPD035"]


print("\n--- BPD035 ---")


print(
    "Age 5-79:",
    int(
        s.between(
            5,
            79
        ).sum()
    )
)


print(
    "80 or older:",
    int(
        s.eq(80)
        .sum()
    )
)


print(
    "Refused (777):",
    int(
        s.eq(777)
        .sum()
    )
)


print(
    "Don't know (999):",
    int(
        s.eq(999)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.between(
            5,
            79
        ).sum()
    )
    == 2130
    and
    int(
        s.eq(80)
        .sum()
    )
    == 41
    and
    int(
        s.eq(777)
        .sum()
    )
    == 1
    and
    int(
        s.eq(999)
        .sum()
    )
    == 40
    and
    int(
        s.isna()
        .sum()
    )
    == 4677
):

    raise ValueError(
        "BPD035 differs from CDC."
    )


print(
    "PASS: BPD035 matches CDC."
)


# ============================================================
# 13. CDC CHECK - BPD058
# ============================================================

#
# How often checked BP at home in past 12 months.
#
# CDC:
#
# 0-1461 = 1,485
# 7777   =     0
# 9999   =     6
# Missing = 5,398
#
# Attached XPT has exactly four corrupted zero
# representations, restored above.
#

s = df["BPD058"]


print("\n--- BPD058 ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "0-1461:",
    int(
        s.between(
            0,
            1461
        ).sum()
    )
)


print(
    "Refused (7777):",
    int(
        s.eq(7777)
        .sum()
    )
)


print(
    "Don't know (9999):",
    int(
        s.eq(9999)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.eq(0)
        .sum()
    )
    == 4
    and
    int(
        s.between(
            0,
            1461
        ).sum()
    )
    == 1485
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 6
    and
    int(
        s.isna()
        .sum()
    )
    == 5398
):

    raise ValueError(
        "BPD058 differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: BPD058 matches CDC."
)


# ============================================================
# 14. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected difference from raw XPT:
#
#     BPD058:
#     four TINY_VALUE values -> numeric 0
#
# NOTHING ELSE may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 21 variables match expected "
    "corrected XPT values exactly."
)


# ============================================================
# 15. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# All BPQ_F variables have been proven whole-number
# variables in this release.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 16. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 17. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 21 CSV columns match "
    "expected corrected XPT values exactly."
)


# ============================================================
# 18. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 19. CHECK CSV FOR REMAINING TINY VALUES
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 20. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# BPQ_F contains NO genuine decimal variables.
#
# Therefore ".0" should not appear anywhere in
# this final CSV.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 21. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 22. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "BPQ_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "BPD058 zero restorations:",
    correction_count
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0: 0 expected"
)


print(
    "Scientific notation: 0 expected"
)

NHANES 2009-2010 BPQ_F XPT -> CSV
Rows: 6,889
Columns: 21
PASS: Structure = 6,889 rows x 21 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 4

Tiny values by variable:
BPD058    4

PASS: Tiny-value pattern matches validated BPQ_F pattern.

--- ZERO RESTORATION ---
BPD058 values restored to zero: 4
PASS: 4 validated BPD058 zeros restored.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in BPQ_F.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: All BPQ_F fields formatted without unnecessary .0.

--- CDC CATEGORICAL FREQUENCY CHECKS ---
PASS: BPQ020 matches CDC.
PASS: BPQ030 matches CDC.
PASS: BPQ040A matches CDC.
PASS: BPQ050A matches CDC.
PASS: BPQ057 matches CDC.
PASS: BPQ056 matches CDC.
PASS: BPQ059 matches CDC.
PASS: BPQ060 matches CDC.
PASS: BPQ070 matches CDC.
PASS: BPQ080 matches CDC.
PASS: BPQ090A matches CDC.
PASS: BPQ090B matches CDC.
PASS: BPQ090C matches CDC.
